# Training on Models

In [5]:
import pandas as pd
import glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
#from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# 1. Load and concatenate all batches
files = glob.glob("data_batches/merged_*.pkl")
frames = []
for fp in files:
    with open(fp, "rb") as f:
        batch = pickle.load(f)

        for sample in batch:
            # assume sample = {"params": array([...]), "results": {"positivity_ok": [...], ...}}
            df_params = pd.DataFrame(sample["params"],                              
                                     columns=["m_phi","m_A","sin_ba","tan_beta","lambda6","lambda7","m12_2"])
            df_labels = pd.DataFrame(sample["results"])
            frames.append(pd.concat([df_params, df_labels], axis=1))
df = pd.concat(frames, ignore_index=True)




In [12]:
print(
    df[['w_h2_bb',
        'w_h2_tautau', 'w_h2_uu', 'w_h2_du', 'w_h2_ln', 'w_h2_gaga',
        'w_h2_Zga', 'w_h2_gg', 'w_h2_hh', 'w_total_h2', 'w_total_top',
        'branching_ratio_h2_gaga', 'lambda1', 'lambda2', 'lambda3', 'lambda4',
        'lambda5', 'lambda6', 'lambda7']].describe() )

        w_h2_bb  w_h2_tautau   w_h2_uu   w_h2_du   w_h2_ln     w_h2_gaga  \
count  975419.0     975419.0  975419.0  975419.0  975419.0  9.754190e+05   
mean        0.0          0.0       0.0       0.0       0.0  1.782666e+06   
std         0.0          0.0       0.0       0.0       0.0  2.228855e+07   
min         0.0          0.0       0.0       0.0       0.0  1.401256e-14   
25%         0.0          0.0       0.0       0.0       0.0  1.117202e-03   
50%         0.0          0.0       0.0       0.0       0.0  4.601048e+01   
75%         0.0          0.0       0.0       0.0       0.0  8.359678e+03   
max         0.0          0.0       0.0       0.0       0.0  1.817245e+09   

           w_h2_Zga   w_h2_gg       w_h2_hh    w_total_h2  ...  \
count  9.754190e+05  975419.0  9.754190e+05  9.754190e+05  ...   
mean   1.454367e+06       0.0  4.750651e+10  4.575321e+11  ...   
std    1.836502e+07       0.0  3.284100e+11  6.553269e+12  ...   
min    9.557640e-15       0.0  0.000000e+00  2.3570

In [6]:
df

,m_phi,m_A,sin_ba,tan_beta,lambda6,lambda7,m12_2,positivity_ok,unitarity_ok,perturbativity_ok,...,w_total_h2,w_total_top,branching_ratio_h2_gaga,lambda1,lambda2,lambda3,lambda4,lambda5,lambda6,lambda7
0,300.028673,300.017321,0.949779,10000.000000,0.100000,0.000000,8.927898,0.0,0.0,0.0,...,5.613000e+06,1.338069,1.028167e-10,-1.079221e+07,0.377818,3647.516628,-0.012070,-0.012070,0.100000,0.000000
1,300.039461,300.064526,0.975726,10000.000000,0.100000,0.000000,9.087525,0.0,0.0,0.0,...,1.514585e+06,1.338069,6.952389e-10,-7.286325e+06,0.316535,2622.613801,0.013794,0.013794,0.100000,0.000000
2,299.979516,299.971754,0.906406,10000.000000,0.100000,0.000000,8.989192,0.0,0.0,0.0,...,1.741102e+07,1.338069,1.008497e-12,-2.172034e+07,0.476502,4697.057184,-0.001508,-0.001508,0.100000,0.000000
3,299.936396,299.943207,0.959960,10000.000000,0.100000,0.000000,8.937705,0.0,0.0,0.0,...,3.698610e+06,1.338069,9.828644e-11,-8.652964e+06,0.353894,3297.912793,-0.009718,-0.009718,0.100000,0.000000
4,299.955198,299.942640,0.978123,10000.000000,0.100000,0.000000,9.052888,0.0,0.0,0.0,...,1.222633e+06,1.338069,3.867892e-10,-6.220333e+06,0.310755,2495.714463,0.009286,0.009286,0.100000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
990395,427.585345,276.808939,0.951799,7706.120660,0.092165,0.055612,5.407232,1.0,0.0,0.0,...,2.261586e+11,1.338069,3.967396e-06,1.284754e+10,0.516986,5990.094961,-214.853191,-214.853191,0.092165,0.055612
990396,457.765264,465.765984,0.984821,1111.646171,0.016959,0.074203,5.678955,1.0,0.0,0.0,...,1.947792e+07,1.338069,3.745529e-06,5.499199e+07,0.353025,570.646549,-44.717945,-44.717945,0.016959,0.074203
990397,431.390907,475.955466,0.960754,5375.343005,-0.099663,-0.044821,0.441003,0.0,0.0,0.0,...,2.304959e+10,1.338069,3.579046e-07,-3.399398e+09,0.473849,4153.918754,116.766332,116.766332,-0.099663,-0.044821
990398,255.376932,479.528292,0.977308,1056.385186,-0.023328,-0.041301,5.735682,0.0,0.0,0.0,...,4.257562e+06,1.338069,4.191402e-07,-2.329594e+07,0.294176,207.448561,18.121886,18.121886,-0.023328,-0.041301


## Constrains con $\lambda_j$

En caso de resolver el potencial completo en su forma generica, es posible obtener rapidamente constrains, reglas que deben cumplirse para evitar perder positividad, unitariedad y perturbatividad.

### Positividad:
$$
\lambda_1 > 0
$$

$$
\lambda_2 > 0
$$

$$
\lambda_3 > - \sqrt{\lambda_1 \lambda_2}
$$

$$
\lambda_3 + \lambda_4 - \lambda_5 > - \sqrt{\lambda_1 \lambda_2}
$$

In [4]:
print("## Missing values per column:")
print(df.isnull().sum(), "\n")

## Missing values per column:
m_phi                          0
m_A                            0
sin_ba                         0
tan_beta                       0
lambda6                        0
lambda7                        0
m12_2                          0
positivity_ok              14981
unitarity_ok               14981
perturbativity_ok          14981
w_h2_bb                    14981
w_h2_tautau                14981
w_h2_uu                    14981
w_h2_du                    14981
w_h2_ln                    14981
w_h2_vv                        0
w_h2_gaga                  14981
w_h2_Zga                   14981
w_h2_gg                    14981
w_h2_hh                    14981
w_total_h2                 14981
w_total_top                14981
branching_ratio_h2_gaga    14981
lambda1                    14981
lambda2                    14981
lambda3                    14981
lambda4                    14981
lambda5                    14981
lambda6                    14981
lambda7      

In [ ]:
df.dropna(axis=0, inplace=True)


In [6]:
# 2. Exploratory Data Analysis

print("## Label distribution:")
print(df[["positivity_ok","perturbativity_ok","unitarity_ok"]].mean(), "\n")

print(df[["positivity_ok","perturbativity_ok","unitarity_ok"]].sum(), "\n")

## Label distribution:
positivity_ok        0.445621
perturbativity_ok    0.000007
unitarity_ok         0.000037
dtype: float64 

positivity_ok        300981.0
perturbativity_ok         5.0
unitarity_ok             25.0
dtype: float64 



In [ ]:

print("## Basic stats for features:")
print(df[].describe(), "\n")


## Basic stats for features:
               m_phi            m_A         sin_ba       tan_beta  \
count  117396.000000  117396.000000  117396.000000  117396.000000   
mean      314.334745     314.985586       0.974996    5006.537637   
std       107.893349     106.790616       0.014434    2883.911465   
min       130.001736     130.004881       0.950000      10.071779   
25%       220.490361     222.514189       0.962500    2510.780852   
50%       310.994780     314.939108       0.974992    5007.539184   
75%       409.513357     407.471695       0.987494    7504.167481   
max       499.999412     499.999452       0.999999    9999.875040   

             lambda6        lambda6        lambda7        lambda7  \
count  117396.000000  117396.000000  117396.000000  117396.000000   
mean        0.000008       0.000008       0.000022       0.000022   
std         0.057732       0.057732       0.057723       0.057723   
min        -0.099999      -0.099999      -0.099999      -0.099999   
25% 